# Train Legal Metrology NER (DistilBERT)

Token-classification model that labels spans:
`product_name`, `mrp`, `net_quantity`, `manufacturer`, `country_of_origin`,
`manufacturing_date`, `expiry_date`.

BIO scheme: **15 tags** (B/I × 7 fields + O).

This notebook is the same path as `python -m app.ml.train_ner`.
It does **not** call OpenAI / Gemini / any hosted LLM.

**Colab T4:** Runtime → GPU. Then:

```
!pip install -r backend/requirements-ml.txt
```

**CPU:** supported, slower. Set `SKIP_NER_TRAIN=1` to skip the heavy fit.

After training, the Flask app uses `EXTRACTOR_MODE=hybrid` (default):
regex first, NER as fallback, conflicts → `ambiguous`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "backend").is_dir():
    BACKEND = ROOT / "backend"
elif (ROOT.parent / "backend").is_dir():
    BACKEND = ROOT.parent / "backend"
else:
    BACKEND = ROOT
sys.path.insert(0, str(BACKEND))
os.chdir(BACKEND)
print("backend:", BACKEND)
print("SKIP_NER_TRAIN:", os.environ.get("SKIP_NER_TRAIN"))
print("EXTRACTOR_MODE:", os.environ.get("EXTRACTOR_MODE", "hybrid"))

## 1. Build the JSONL dataset

Gold + DemoMart samples + 300–600 paraphrases of the same fields.
Split is **by document** 70/15/15. Regex-blind SKUs stay in TEST.

In [ ]:
from app.ml.make_dataset import build_dataset, GOLD_DIR

counts = build_dataset()
print(counts)
print("files:", list(GOLD_DIR.glob("*.jsonl")))

## 2. Train DistilBERT token classifier

`distilbert-base-uncased` + `AutoModelForTokenClassification`.
Pass `multilingual=True` to switch to `bert-base-multilingual-cased`.

max_len=256, stride=64. HuggingFace Trainer. Seqeval entity P/R/F1 on TEST.

In [ ]:
from app.ml.train_ner import CPU_FALLBACK_NOTE, main as train_main
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(CPU_FALLBACK_NOTE)

# Same CLI: python -m app.ml.train_ner
# Add --multilingual for bert-base-multilingual-cased
rc = train_main([])
print("train_ner exit", rc)

## 3. Read RESULTS.md

Three rows on the **same** test texts: `regex_only` | `ner_only` | `hybrid`.
If hybrid does not beat regex recall on manufacturer / origin / dates, the file says so.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

results = Path("app/ml/RESULTS.md")
if results.is_file():
    display(Markdown(results.read_text(encoding="utf-8")))
else:
    print("RESULTS.md not written yet — train first.")